## 📝 실전 문제풀이 Set 3: 시중 스마트폰 상세 정보

### 1) 데이터 및 시나리오

▶ 신입사원 김모씨는 신규 스마트폰 스펙 기획 업무를 보조하기 위해 온라인 디지털 마켓 사이트에서 판매 중인 스마트폰 데이터를 수집하여 이를 분석하고 백데이터로 활용하기로 결심하였다.

* **데이터 개요:** `Xa mobiles.csv` (430 rows, 11 columns, UTF-8)

**[변수 상세]**

| 변수명 | 유형 | 설명 |
| --- | --- | --- |
| `screen_size` | string | 화면 크기 |
| `ROM` | int | 저장 공간 용량 |
| `RAM` | int | RAM 용량 |
| `num_rear_camera` | int | 후면 카메라 개수 |
| `num_front_camera` | int | 전면 카메라 개수 |
| `battery_capacity` | int | 배터리 용량 |
| `ratings` | float | 평가 점수 평균 |
| `num_of_ratings` | int | 평가 개수 |
| `sales_price` | int | 판매가격 |
| `discount_percent` | float | 할인율 |
| `sales` | float | 판매 지수 |


## 2) 문제

* **필요 라이브러리:** `MinMaxScaler`, `train_test_split`, `KNeighborsRegressor`, `mean_squared_error`

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

In [6]:
df = pd.read_csv("../dataset/mobiles.csv")
df.head(10)

,screen_size,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,sales
0,Very Small,64,2,1,1,1800,4.5,38645,32999,0.17,127.52
1,Small,64,4,2,1,2815,4.5,244,57149,0.04,1.39
2,Very Small,64,2,1,1,1800,4.5,38645,32999,0.17,127.52
3,Medium,64,3,1,1,2942,4.6,5366,42999,0.10,23.07
4,Medium,128,4,2,1,2815,4.6,745,69149,0.02,5.15
5,Medium,64,4,2,1,2815,4.6,745,64149,0.02,4.78
6,Medium,128,4,2,1,2815,4.6,745,69149,0.02,5.15
7,Medium,64,4,2,1,2815,4.6,745,64149,0.02,4.78
8,Medium,128,4,2,1,2815,4.6,745,69149,0.02,5.15
9,Medium,128,4,2,1,2815,4.6,745,69149,0.02,5.15


### **Q01.** 판매지수(sales)를 기준으로 이상치라고 판단되는 제품을 '주목받는 제품'이라고 판단하고 해당 제품들의 성능지표를 산출하시오. **산출된 성능지표의 평균은 얼마인가?**

* `<성능지표 계산식>`:  $E = \frac{ROM}{32} + \frac{RAM}{2} + \text{카메라 개수} + \frac{battery\_capacity}{1000}$
* ※ 카메라 개수 = num_rear_camera + num_front_camera
* ※ 이상치는 평균으로부터 2표준편차보다 큰 값으로 정의한다.
* ※ 결과는 반올림하여 소수점 둘째 자리까지 계산하시오. (정답 예시: 0.12)

In [36]:
df_q1 = df.copy()
q1_sales = df_q1['sales']
q1_sales_mean = q1_sales.mean()
q1_sales_std = q1_sales.std()
q1_sales_outfitter = q1_sales_mean + 2 * q1_sales_std

q1_focus = df_q1.loc[df_q1['sales'] > q1_sales_outfitter, ]
q1_focus_E = q1_focus['ROM']/32 + q1_focus['RAM']/2 + q1_focus['num_rear_camera'] + q1_focus['num_front_camera'] + q1_focus['battery_capacity']/1000
round(q1_focus_E.mean(),2)

11.01

### Q1.

In [15]:
stat_mean = df["sales"].mean()
stat_std  = df["sales"].std()
stat_out  = stat_mean + 2 * stat_std
stat_out

146.5515012927322

In [16]:
df_q1 = df.loc[df["sales"] > stat_out, ]

In [17]:
len(df_q1)

16

In [18]:
df_q1.head(1)

,screen_size,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,sales
98,Medium,128,6,2,1,4000,4.6,122001,18999,0.09,231.79


In [19]:
df_q1["idx"] = (df_q1["ROM"] / 32) + (df_q1["RAM"] / 2) + (df_q1["num_front_camera"] + df_q1["num_rear_camera"])
df_q1["idx"] = df_q1["idx"] + (df_q1["battery_capacity"] / 1000)

x:\study\docs\data_science\.venv\lib\site-packages\ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
x:\study\docs\data_science\.venv\lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [20]:
round(df_q1["idx"].mean(), 2)

11.01


### **Q02.** 판매 지수(sales)와 가장 상관관계가 높은 변수를 찾고자 한다. 배터리 용량, 평가 점수 평균, 평가 개수, 판매 가격, 할인율 변수와 판매 지수를 피어슨 상관분석을 실시하였을 때, **상관계수의 절대값이 가장 큰 변수의 상관계수는 얼마인가?**

* ※ 후면 카메라가 1개인 제품은 제외하시오.
* ※ 결과는 반올림하여 소수점 둘째 자리까지 출력하시오. (정답 예시: 0.12)

## .corr(method='pearson')['sales'].drop('sales')

In [44]:
df_q2 = df.copy()
df_q2_1 = df_q2.loc[df_q2['num_rear_camera'] != 1, :]
corr_cols = [
    'battery_capacity', 'ratings', 'num_of_ratings',
    'sales_price', 'discount_percent', 'sales'
]

df_q2_1_corr = df_q2_1[corr_cols].corr(method='pearson')['sales'].drop('sales')
display(df_q2_1_corr)

df_q2_1_corr_idxmax = df_q2_1_corr.abs().idxmax()
display(df_q2_1_corr_idxmax)

df_q2_result = round(df_q2_1_corr[df_q2_1_corr_idxmax],2)
display(df_q2_result)

battery_capacity    0.025680
ratings             0.226075
num_of_ratings      0.949114
sales_price        -0.247760
discount_percent    0.223471
Name: sales, dtype: float64

'num_of_ratings'

0.95

### Q2.

In [13]:
df["num_rear_camera"].unique()

array([1, 2, 3, 4], dtype=int64)

In [ ]:
df_q2 = df.loc[df["num_rear_camera"] != 1, "battery_capacity":].reset_index(drop = True)
df_q2.head(1)

In [16]:
df_corr = df_q2.corr()
type(df_corr)

pandas.core.frame.DataFrame

In [ ]:
df_corr["sales"].abs().round(2) # 0.95

In [20]:
from scipy.stats import pearsonr

In [25]:
stat1, p = pearsonr(df_q2["sales"], df_q2["battery_capacity"]) 
stat2, p = pearsonr(df_q2["sales"], df_q2["ratings"]) 
stat3, p = pearsonr(df_q2["sales"], df_q2["num_of_ratings"]) 
stat4, p = pearsonr(df_q2["sales"], df_q2["sales_price"]) 
stat5, p = pearsonr(df_q2["sales"], df_q2["discount_percent"]) 
round(max([stat1, stat2, stat3, stat4, stat5]), 2)

0.95

### **Q03.** 판매 지수(sales)를 예측하기 위해 k-NN 알고리즘을 사용하고, 이웃의 개수를 변화하면서 가장 성능이 좋은 모델을 확보하려 한다. **RMSE를 기준으로 가장 성능이 좋은 모델을 확인하고 해당 모델의 k(이웃 개수)를 구하라.**

* `<독립변수>`: 판매 지수를 제외한 모든 변수
* `<종속변수>`: 판매 지수
* ※ 명목형 독립변수는 One Hot Encoding을 실시하시오. (학습에 사용하는 독립변수 개수는 14개)
* ※ 분할비는 8:2, Min-Max 정규화 실시, seed 123.
* ※ 최근접 이웃은 3, 5, 7, 9, 11개를 사용하시오. (정답 예시: 7)

### Q3.

In [50]:
df_q3 = df.copy()

X = pd.get_dummies(df_q3.drop(columns = 'sales'), drop_first=False)
y = df_q3['sales']

#분할비 8:2
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 123)

#정규화 --> 독립변수
scaler = MinMaxScaler()
X_trained_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#모델링
rmse_k = {}
for k in [3,5,7,9,11]:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_trained_scaled, y_train)
    pred = knn.predict(X_test_scaled)
    rmse_k[k] = mean_squared_error(y_test, pred) ** 0.5

#pd 계산이 쉽게 pd로 변환
series_rmse = pd.Series(rmse_k, name='RMSE')
series_rmse.idxmin(), series_rmse.min()


(3, 40.440548901789604)

`pd.get_dummies()`는 다른 메서드/함수/클래스와 다르게 "columns" 인자에 단일 값을 할당하는 경우에도 반드시 🌟**리스트**🌟 객체를 사용하여 할당해야 한다. 단순 문자열을 할당할 경우 에러가 난다.

그리고 원핫인코딩을 실시할 때 변수명에 띄어쓰기가 있을 수 있는데 `statsmodels` 라이브러리 기반 모델링을 하면서 formula 를 사용하는 경우 변수명에 띄어쓰기를 제거하지 않은 채로 formula를 작성하면 반드시 에러가 발생함. 그리고 이 이슈는 이전 시험에서 응시자가 어려움을 겪은 사례가 있음.  
※ 다음의 코드 결과에서는 "screen_size_Very Large"  
※ "screen_size_Very Large" -> "screen_size_Very_Large"

In [51]:
# df_q3_dum = pd.get_dummies(df, columns = ["screen_size"]) # 시험버전
df_q3_dum = pd.get_dummies(df, columns = ["screen_size"], dtype = "int") # 최신버전
df_q3_dum.head(2)

,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,sales,screen_size_Large,screen_size_Medium,screen_size_Small,screen_size_Very Large,screen_size_Very Small
0,64,2,1,1,1800,4.5,38645,32999,0.17,127.52,0,0,0,0,1
1,64,4,2,1,2815,4.5,244,57149,0.04,1.39,0,0,1,0,0


In [52]:
df_q3_dum.columns.str.replace(" ", "_")

Index(['ROM', 'RAM', 'num_rear_camera', 'num_front_camera', 'battery_capacity',
       'ratings', 'num_of_ratings', 'sales_price', 'discount_percent', 'sales',
       'screen_size_Large', 'screen_size_Medium', 'screen_size_Small',
       'screen_size_Very_Large', 'screen_size_Very_Small'],
      dtype='object')

In [53]:
df_q3_dum = df_q3_dum.set_index("sales").reset_index()
df_q3_dum.head(1)

,sales,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,screen_size_Large,screen_size_Medium,screen_size_Small,screen_size_Very Large,screen_size_Very Small
0,127.52,64,2,1,1,1800,4.5,38645,32999,0.17,0,0,0,0,1


In [54]:
df_q3_dum.shape

(430, 15)

In [55]:
df_train, df_test = train_test_split(df_q3_dum, train_size = 0.8, random_state = 123)
len(df_train), len(df_test)

(344, 86)

In [56]:
model_nor = MinMaxScaler().fit(df_train)
arr_train_nor = model_nor.transform(df_train)
arr_test_nor  = model_nor.transform(df_test)

In [57]:
arr_train_nor[:1, ]

array([[0.00394753, 0.04761905, 0.18181818, 0.33333333, 0.        ,
        0.42307692, 0.625     , 0.00396262, 0.02607261, 0.09302326,
        0.        , 0.        , 1.        , 0.        , 0.        ]])

In [58]:
ls_k = [3, 5, 7, 9, 11]
k = ls_k[0]

model_knn = KNeighborsRegressor(n_neighbors = k)
model_knn.fit(X = arr_train_nor[:, 1:],
              y = arr_train_nor[:, 0])
pred = model_knn.predict(arr_test_nor[:, 1:])
mean_squared_error(y_true = arr_test_nor[:, 0], y_pred = pred) ** 0.5

0.08186677375964535

In [59]:
ls_k = [3, 5, 7, 9, 11]
ls_rmse = []
for k in ls_k:
    model_knn = KNeighborsRegressor(n_neighbors = k)
    model_knn.fit(X = arr_train_nor[:, 1:],
                  y = arr_train_nor[:, 0])
    pred = model_knn.predict(arr_test_nor[:, 1:])
    val_rmse = mean_squared_error(y_true = arr_test_nor[:, 0], y_pred = pred) ** 0.5
    ls_rmse = ls_rmse + [val_rmse]

In [60]:
ser_rmse = pd.Series(ls_rmse, index = ls_k)
val_best_k = ser_rmse.idxmin()
val_best_k, ser_rmse.min()

(3, 0.08186677375964535)

### Q3. \[추가 지시사항\] 다음은 저번달에 신규 출시된 경쟁사의 스마트폰 정보이다. 해당 스마트폰의 판매지수는 얼마로 예상되는가?  
> **단계 1)** 주어진 데이터는 기존 원핫인코딩 규칙을 기반으로 더미변수를 생성하시오.  
> **단계 2)** 기존 정규화 규칙을 기반으로 주어진 데이터를 정규화 하시오.  
> **단계 3)** 기존에 정제한 학습 데이터 세트와 이웃 개수는 직전에 최적이라고 판단한 k값을 사용한 k-NN 모델을 준비하시오.  
> **단계 4)** 준비된 k-NN 모델에 "단계 2"에서 산출한 데이터 세트를 입력하고 그 결과를 확인하시오.  
> **단계 5)** "단계 4"의 결과물을 기존 정규화 규칙을 기반으로 역변환 하시오.  

※ 정답은 반올림하여 소수 첫째 자리까지 출력하시오.  
(정답 예시: 0.1)
* ROM: 256
* RAM: 6
* num_rear_camera: 4
* num_front_camera: 1
* battery_capacity: 4000
* ratings: 4.3
* num_of_ratings: 25000
* sales_price: 85000
* discount_percent: 0.05
* screen_size: "Large"


In [ ]:
# df_t1 = pd.DataFrame(dict(ROM = 256, RAM = 6)) # ❌❌
df_t1 = pd.DataFrame(dict(ROM = [256], RAM = [6]))
df_t1

In [61]:
df_t1 = df_test.head(1).reset_index(drop = True)
df_t1["RAM"] = 6
df_t1["num_rear_camera"] = 4
df_t1["num_front_camera"] = 1
df_t1["battery_capacity"] = 4000
df_t1["ratings"] = 4.3
df_t1["num_of_ratings"] = 25000
df_t1["sales_price"] = 85000
df_t1["discount_percent"] = 0.05
df_t1["screen_size_Large"] = 1
df_t1["screen_size_Medium"] = 0

In [62]:
df_t1

,sales,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,screen_size_Large,screen_size_Medium,screen_size_Small,screen_size_Very Large,screen_size_Very Small
0,5.9,256,6,4,1,4000,4.3,25000,85000,0.05,1,0,0,0,0


In [ ]:
model_nor.data_max_

In [ ]:
# arr_t1_nor = model_nor.transform(df_t1.drop(columns = "sales")) # 에러
arr_t1_nor = model_nor.transform(df_t1)
arr_t1_nor

In [ ]:
model_knn_b = KNeighborsRegressor(n_neighbors = val_best_k)
model_knn_b.fit(X = arr_train_nor[:, 1:],
                y = arr_train_nor[:, 0])

In [72]:
pred_t1 = model_knn_b.predict(arr_t1_nor[:, 1:])
pred_t1

array([0.00132259])

In [78]:
arr_t1_nor[0, 0] = pred_t1
arr_t1_nor

array([[0.00132259, 0.49206349, 0.45454545, 1.        , 0.        ,
        0.42307692, 0.625     , 0.05308122, 0.51815842, 0.09302326,
        1.        , 0.        , 0.        , 0.        , 0.        ]])

In [82]:
# model_nor.inverse_transform(pred_t1)
# model_nor.inverse_transform([pred_t1])
arr_t1_inv = model_nor.inverse_transform(arr_t1_nor)
arr_t1_inv[0, 0]

0.6533333333333334

In [83]:
df_t1_inv = pd.DataFrame(arr_t1_inv, columns = df_t1.columns)
df_t1_inv

,sales,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,screen_size_Large,screen_size_Medium,screen_size_Small,screen_size_Very Large,screen_size_Very Small
0,0.653333,256.0,6.0,4.0,1.0,4000.0,4.3,25000.0,85000.0,0.05,1.0,0.0,0.0,0.0,0.0


In [84]:
df_t1

,sales,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,screen_size_Large,screen_size_Medium,screen_size_Small,screen_size_Very Large,screen_size_Very Small
0,5.9,256,6,4,1,4000,4.3,25000,85000,0.05,1,0,0,0,0
